# 批跑 + 全地点汇总 (NEW pipeline)

单场景精跑用 `run_rider_count_new.ipynb`;本 notebook 负责:**批量跑完所有未排除地点 → 全地点汇总表 → 与人工计数对照 → 复查CSV指标**。

支持断点续跑:已完成的地点自动跳过(FORCE_RERUN=True 强制重跑)。

In [ ]:
# Cell 1: Bootstrap
import sys, importlib.util
from pathlib import Path

p = Path.cwd().resolve()
while p != p.parent and not (p / "src").exists():
    p = p.parent
REPO_ROOT = p
sys.path.insert(0, str(REPO_ROOT))
script = REPO_ROOT / "scripts" / "run_rider_count_new.py"
if not script.exists():
    hits = list(REPO_ROOT.rglob("run_rider_count_new.py"))
    assert hits, "找不到 run_rider_count_new.py"
    script = hits[0]
spec = importlib.util.spec_from_file_location("run_rider_count_new", script)
rrc = importlib.util.module_from_spec(spec)
sys.modules["run_rider_count_new"] = rrc
spec.loader.exec_module(rrc)
print("Repo root:", REPO_ROOT, "| 脚本:", script)

In [ ]:
# Cell 2: 配置
from types import SimpleNamespace

DATA_ROOT = Path(r"D:\0_MAIN_BIKE_DATASETS_clean")
OUT_ROOT  = REPO_ROOT / "outputs_new"
CFG_DIR   = REPO_ROOT / "configs" / "locations_new"

# 排除的场景 (按 config 命名; 可自行增删)
EXCLUDE = {"loc_01", "loc_03", "loc_05", "loc_05-2",
           "loc_07", "loc_07-2", "loc_09", "loc_20", "loc_22"}

FORCE_RERUN = False   # True = 已跑过的也重跑

args = SimpleNamespace(
    model="yolov8s.pt", imgsz=1280, conf=0.10, classes={1},
    nms_iou=0.70, assoc_gap=3, min_move_px=25.0, cos_gate=0.5,
    save_crops=True, save_viz=True, max_images=None,
)
todo = [c.stem for c in sorted(CFG_DIR.glob("loc_*.json"))
        if "old" not in c.stem.lower() and c.stem not in EXCLUDE]
print(f"待跑 {len(todo)} 个地点:", ", ".join(todo))

In [ ]:
# Cell 3: 批量跑 (断点续跑; 挂机即可)
done, skipped, failed = [], [], []
for loc in todo:
    outdir = OUT_ROOT / loc
    if not FORCE_RERUN and (outdir / "scene_summary.json").exists():
        skipped.append(loc); continue
    img_dir = rrc.find_img_dir(DATA_ROOT, loc)
    if img_dir is None:
        print(f"[{loc}] 找不到图片文件夹, 跳过"); failed.append(loc); continue
    try:
        rrc.run_location(loc, img_dir, CFG_DIR / f"{loc}.json", outdir, args)
        done.append(loc)
    except Exception as e:
        print(f"[{loc}] 失败: {e}"); failed.append(loc)
print(f"\n本次跑完 {len(done)} | 已存在跳过 {len(skipped)} | 失败 {failed or '无'}")

In [ ]:
# Cell 4: 全地点汇总表
agg = rrc.aggregate_locations(OUT_ROOT, exclude=EXCLUDE)
agg.to_csv(OUT_ROOT / "all_locations_summary.csv", index=False)
display(rrc.all_locations_table(agg))
print("已存:", OUT_ROOT / "all_locations_summary.csv")

In [ ]:
# Cell 5: 与人工计数对照 (把你的手动结果填进来: 地点: (顺, 逆))
MANUAL = {
    "loc_04":   (179, 62),
    "loc_04-2": (117, 45),
    # "loc_02": (?, ?),  ← 按此格式继续添加
}
display(rrc.manual_comparison_table(agg, MANUAL))
print("ΔWW 红底 = 偏差超过 8 个百分点, 优先诊断这些地点")

In [ ]:
# Cell 6: 汇总图 — 各地点 WW 率 (带CI) 与设施分布
import matplotlib.pyplot as plt
import numpy as np
rrc._mpl_style()
d = agg[agg.dir_known > 0].sort_values("ww_rate", ascending=True)
fig, ax = plt.subplots(figsize=(9, 0.4*len(d)+1.5))
y = np.arange(len(d))
err = np.array([ (d.ww_rate-d.ww_lo).tolist(), (d.ww_hi-d.ww_rate).tolist() ])
ax.barh(y, d.ww_rate, xerr=err, height=0.55, color="#2a78d6",
        error_kw=dict(ecolor="#52514e", lw=1.4, capsize=3))
for i,(_,r) in enumerate(d.iterrows()):
    ax.text(min(r.ww_hi+0.03,1.0), i, f"{r.against}/{r.dir_known}", va="center", fontsize=9, color="#52514e")
ax.set_yticks(y); ax.set_yticklabels(d.location)
ax.set_xlim(0,1); ax.set_xlabel("wrong-way rate (95% CI)")
ax.set_title("Wrong-way rate by location — displacement method (gated only)")
ax.grid(axis="y", visible=False)
plt.tight_layout(); plt.show()

In [ ]:
# Cell 7: 复查CSV指标汇总 (每标注完一个场景的 review 页就重跑本cell)
import pandas as pd
rows = []
for d_ in sorted(OUT_ROOT.glob("loc_*")):
    res = rrc.analyze_review_csv(d_, d_.name)
    if res: rows.append(res)
if rows:
    rev = pd.DataFrame(rows)
    rev_display = rev.assign(**{
        "precision": rev.precision.map("{:.0%}".format),
        "manual_ww": rev.manual_ww.map("{:.1%}".format),
        "corrected_ww": rev.corrected_ww.map("{:.1%}".format)})
    display(rev_display)
    rev.to_csv(OUT_ROOT / "review_metrics.csv", index=False)
else:
    print("还没有已标注的 review_*.csv — 用单场景notebook的Cell 6生成复查页并标注导出")